In [7]:
import os

RAW_DIR = r"D:\Fir Document\Universitas Negeri Surabaya\Semester 4\Data Warehouse\Task\Final_Project\GemDataEXTR"

if os.path.exists(RAW_DIR):
    print(f"✓ Folder ditemukan: {RAW_DIR}")
    print(f"\nIsi folder:")
    for f in sorted(os.listdir(RAW_DIR)):
        size = os.path.getsize(os.path.join(RAW_DIR, f))
        print(f"  {repr(f)} ({size:,} bytes)")
else:
    print(f"✗ Folder tidak ditemukan: {RAW_DIR}")
    print("Cek ulang path-nya")

✓ Folder ditemukan: D:\Fir Document\Universitas Negeri Surabaya\Semester 4\Data Warehouse\Task\Final_Project\GemDataEXTR

Isi folder:
  'CPI Price, % y-o-y, median weighted, seas. adj..xlsx' (52,603 bytes)
  'CPI Price, % y-o-y, nominal, seas. adj..xlsx' (401,730 bytes)
  'CPI Price, nominal, not seas. adj..xlsx' (541,318 bytes)
  'CPI Price, nominal, seas. adj..xlsx' (422,981 bytes)
  'Core CPI, not seas. adj..xlsx' (269,425 bytes)
  'Core CPI, seas. adj..xlsx' (253,696 bytes)
  'Exchange rate, new LCU per USD extended backward, period average.xlsx' (630,202 bytes)
  'Exchange rate, old LCU per USD extended forward, period average.xlsx' (621,031 bytes)
  'Exports Merchandise, Customs, Price, US$, not seas. adj..xlsx' (210,804 bytes)
  'Exports Merchandise, Customs, Price, US$, seas. adj..xlsx' (225,368 bytes)
  'Exports Merchandise, Customs, constant 2010 US$, millions, not seas. adj..xlsx' (244,958 bytes)
  'Exports Merchandise, Customs, constant 2010 US$, millions, seas. adj..xlsx' 

In [8]:
import pandas as pd
import os

RAW_DIR = r"D:\Fir Document\Universitas Negeri Surabaya\Semester 4\Data Warehouse\Task\Final_Project\GemDataEXTR"
OUT_DIR = r"D:\Fir Document\Universitas Negeri Surabaya\Semester 4\Data Warehouse\Task\Final_Project\dwh-energy-project\data\raw_csv"

os.makedirs(OUT_DIR, exist_ok=True)
print(f"Output folder: {OUT_DIR}")

TARGET_COUNTRIES = [
    "United States", "China", "Germany", "Japan",
    "India", "United Kingdom", "France", "Brazil"
]

FILES = {
    "CPI Price, % y-o-y, nominal, seas. adj..xlsx"                        : "wb_cpi.csv",
    "GDP at market prices, current US$, millions, seas. adj..xlsx"         : "wb_gdp.csv",
    "Industrial Production, constant 2010 US$, seas. adj..xlsx"            : "wb_industrial.csv",
    "Unemployment Rate, seas. adj..xlsx"                                    : "wb_unemployment.csv",
    "Exchange rate, new LCU per USD extended backward, period average.xlsx" : "wb_exchange_rate.csv",
}

def convert_wb_excel(xlsx_name, csv_name):
    path = os.path.join(RAW_DIR, xlsx_name)
    
    if not os.path.exists(path):
        print(f"✗ File tidak ditemukan: {xlsx_name}")
        return False
    
    xl    = pd.ExcelFile(path)
    sheet = "monthly" if "monthly" in xl.sheet_names else xl.sheet_names[0]
    print(f"\n{xlsx_name[:55]}...")
    print(f"Sheet : '{sheet}'")
    
    df = pd.read_excel(path, sheet_name=sheet, header=0, index_col=0)
    df = df.dropna(how="all")
    
    # Filter negara target
    available = [c for c in TARGET_COUNTRIES if c in df.columns]
    missing   = [c for c in TARGET_COUNTRIES if c not in df.columns]
    df = df[available]
    
    if missing:
        print(f"   ⚠ Negara tidak ada di file ini: {missing}")
    print(f"   Negara  : {available}")
    
    # Filter periode 2022-2024
    def in_range(idx):
        return any(str(y) in str(idx) for y in ["2022", "2023", "2024"])
    
    df = df[df.index.map(in_range)]
    df = df.reset_index()
    df.columns = ["period_raw"] + available
    
    out_path = os.path.join(OUT_DIR, csv_name)
    df.to_csv(out_path, index=False)
    print(f"   Shape   : {df.shape}")
    print(f"   Period  : {df['period_raw'].iloc[0]} → {df['period_raw'].iloc[-1]}")
    print(f"   ✓ Tersimpan: {csv_name}")
    return True

# Jalankan konversi semua file
print("Convertion EXCEL to CSV")

success = 0
for xlsx_name, csv_name in FILES.items():
    if convert_wb_excel(xlsx_name, csv_name):
        success += 1

print(f"\n{'='*60}")
print(f"Selesai: {success}/{len(FILES)} file berhasil dikonversi")
print(f"\nHasil di folder output:")
for f in sorted(os.listdir(OUT_DIR)):
    size = os.path.getsize(os.path.join(OUT_DIR, f))
    print(f"  ✓ {f} ({size:,} bytes)")

Output folder: D:\Fir Document\Universitas Negeri Surabaya\Semester 4\Data Warehouse\Task\Final_Project\dwh-energy-project\data\raw_csv
Convertion EXCEL to CSV

CPI Price, % y-o-y, nominal, seas. adj..xlsx...
Sheet : 'monthly'
   Negara  : ['United States', 'China', 'Germany', 'Japan', 'India', 'United Kingdom', 'France', 'Brazil']
   Shape   : (36, 9)
   Period  : 2022M01 → 2024M12
   ✓ Tersimpan: wb_cpi.csv

GDP at market prices, current US$, millions, seas. adj....
Sheet : 'annual'
   Negara  : ['United States', 'China', 'Germany', 'Japan', 'India', 'United Kingdom', 'France', 'Brazil']
   Shape   : (3, 9)
   Period  : 2022.0 → 2024.0
   ✓ Tersimpan: wb_gdp.csv

Industrial Production, constant 2010 US$, seas. adj..xl...
Sheet : 'monthly'
   Negara  : ['United States', 'China', 'Germany', 'Japan', 'India', 'United Kingdom', 'France', 'Brazil']
   Shape   : (36, 9)
   Period  : 2022M01 → 2024M12
   ✓ Tersimpan: wb_industrial.csv

Unemployment Rate, seas. adj..xlsx...
Sheet : 'monthly'

In [9]:
print("PREVIEW SETIAP FILE CSV")
print("=" * 60)

for fname in sorted(os.listdir(OUT_DIR)):
    if not fname.endswith(".csv"):
        continue
    
    fpath = os.path.join(OUT_DIR, fname)
    df    = pd.read_csv(fpath)
    
    print(f"\n📄 {fname}")
    print(f"   Shape  : {df.shape}")
    print(f"   Kolom  : {df.columns.tolist()}")
    if "period_raw" in df.columns:
        print(f"   Period : {df['period_raw'].iloc[0]} → {df['period_raw'].iloc[-1]}")
    print(df.head(3).to_string())

PREVIEW SETIAP FILE CSV

📄 wb_cpi.csv
   Shape  : (36, 9)
   Kolom  : ['period_raw', 'United States', 'China', 'Germany', 'Japan', 'India', 'United Kingdom', 'France', 'Brazil']
   Period : 2022M01 → 2024M12
  period_raw  United States     China   Germany     Japan     India  United Kingdom    France    Brazil
0    2022M01       7.558806  0.832049  4.142012  0.501505  6.104194        4.939261  2.918622  10.50204
1    2022M02       7.937279  1.131222  4.322200  1.002004  6.081731        5.528443  3.612800  10.56309
2    2022M03       8.572205  1.634121  5.876592  1.200000  6.866387        6.302448  4.545975  11.31383

📄 wb_exchange_rate.csv
   Shape  : (36, 9)
   Kolom  : ['period_raw', 'United States', 'China', 'Germany', 'Japan', 'India', 'United Kingdom', 'France', 'Brazil']
   Period : 2022M01 → 2024M12
  period_raw  United States     China   Germany     Japan     India  United Kingdom    France    Brazil
0    2022M01            1.0  6.355984  0.883187  114.8574  74.48129        0.7

In [10]:
SERVER     = "inter24@178.128.52.238"
SERVER_DIR = "/home/inter24/dags/energy_dwh/raw/"

print("Jalankan perintah berikut di PowerShell untuk upload ke server:")
print("=" * 60)

for fname in sorted(os.listdir(OUT_DIR)):
    if not fname.endswith(".csv"):
        continue
    local_path = os.path.join(OUT_DIR, fname)
    print(f'\nscp "{local_path}" {SERVER}:{SERVER_DIR}')

Jalankan perintah berikut di PowerShell untuk upload ke server:

scp "D:\Fir Document\Universitas Negeri Surabaya\Semester 4\Data Warehouse\Task\Final_Project\dwh-energy-project\data\raw_csv\wb_cpi.csv" inter24@178.128.52.238:/home/inter24/dags/energy_dwh/raw/

scp "D:\Fir Document\Universitas Negeri Surabaya\Semester 4\Data Warehouse\Task\Final_Project\dwh-energy-project\data\raw_csv\wb_exchange_rate.csv" inter24@178.128.52.238:/home/inter24/dags/energy_dwh/raw/

scp "D:\Fir Document\Universitas Negeri Surabaya\Semester 4\Data Warehouse\Task\Final_Project\dwh-energy-project\data\raw_csv\wb_gdp.csv" inter24@178.128.52.238:/home/inter24/dags/energy_dwh/raw/

scp "D:\Fir Document\Universitas Negeri Surabaya\Semester 4\Data Warehouse\Task\Final_Project\dwh-energy-project\data\raw_csv\wb_industrial.csv" inter24@178.128.52.238:/home/inter24/dags/energy_dwh/raw/

scp "D:\Fir Document\Universitas Negeri Surabaya\Semester 4\Data Warehouse\Task\Final_Project\dwh-energy-project\data\raw_csv\wb_u